# 01. 환경 확인

실험 시작 전 체크리스트:
1. GPU / CUDA 환경
2. 패키지 버전
3. 모델 로드 & 프롬프트 포맷 (Qwen3-4B 핵심)
4. KV 캐시 압축 동작
5. LongBench 데이터셋 로드
6. 메트릭 계산
7. End-to-End 단일 샘플 테스트

In [ ]:
import sys, os
# 프로젝트 루트를 path에 추가
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('Working dir:', os.getcwd())

## 1. GPU / CUDA 환경

In [ ]:
import torch

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        total_gb = prop.total_memory / 1024**3
        free_gb  = (prop.total_memory - torch.cuda.memory_reserved(i)) / 1024**3
        print(f'  GPU {i}: {prop.name}  Total={total_gb:.1f}GB  Free={free_gb:.1f}GB')
else:
    print('WARNING: CUDA not available')

## 2. 패키지 버전 확인

In [ ]:
import importlib

packages = {
    'torch': '2.1.0',
    'transformers': '4.38.0',
    'datasets': '>=2.14',
    'numpy': '>=1.24',
    'accelerate': '>=0.25',
}

print(f'{"Package":<20} {"Installed":<15} {"Required":<15} OK?')
print('-' * 60)
for pkg, req in packages.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        ok  = 'v' if ver != 'unknown' else '?'
    except ImportError:
        ver = 'NOT INSTALLED'
        ok  = 'X'
    print(f'{pkg:<20} {ver:<15} {req:<15} {ok}')

## 3. 모델 로드 & 프롬프트 포맷 확인

**Qwen3-4B 핵심 체크포인트:**
- `enable_thinking=False` 적용 여부 (`<think>` 토큰 없어야 함)
- `pad_token` 설정 여부 (없으면 token embedding 오류 발생)

In [ ]:
MODEL_KEY = 'qwen3-4b'  # 변경 가능: 'phi-3-mini' | 'gemma-2-2b'

from core.model_loader import load_model_and_tokenizer, make_prompt, tokenize_prompt, MODEL_CONFIGS
print('모델 설정:')
for k, v in MODEL_CONFIGS[MODEL_KEY].items():
    print(f'  {k}: {v}')

In [ ]:
# 모델 로드 (2~5분 소요)
model, tokenizer, model_config = load_model_and_tokenizer(
    model_key=MODEL_KEY,
    device='cuda',
    use_flash_attn=False,
)
print('Model loaded!')
print(f'  num_layers   = {model_config["num_layers"]}')
print(f'  num_heads    = {model_config["num_heads"]}')
print(f'  num_kv_heads = {model_config["num_kv_heads"]}  (GQA={model_config["num_heads"] != model_config["num_kv_heads"]})')
print(f'  dtype        = {model_config["dtype"]}')

In [ ]:
# pad_token 확인
print(f'pad_token    : {repr(tokenizer.pad_token)}')
print(f'pad_token_id : {tokenizer.pad_token_id}')
print(f'eos_token    : {repr(tokenizer.eos_token)}')

if tokenizer.pad_token is None:
    print('ERROR: pad_token is None! token embedding errors will occur.')
else:
    print('OK: pad_token is set')

In [ ]:
# 프롬프트 포맷 확인
context  = 'The Eiffel Tower is located in Paris, France. Built in 1889 by Gustave Eiffel.'
question = 'Where is the Eiffel Tower?'

prompt = make_prompt(MODEL_KEY, tokenizer, context, question, task_type='qa')
print('--- Generated Prompt ---')
print(repr(prompt[:500]))
print()

# Qwen3 전용 체크
if MODEL_KEY == 'qwen3-4b':
    if '<think>' in prompt:
        print('ERROR: <think> token found! enable_thinking=False is not working.')
    else:
        print('OK: No <think> token (enable_thinking=False working)')
    if '<|im_start|>' in prompt:
        print('OK: chat_template applied (<|im_start|> found)')
    else:
        print('WARNING: chat_template may not be applied')

In [ ]:
# 토크나이즈 확인
inputs = tokenize_prompt(prompt, tokenizer, MODEL_KEY, max_input_length=31000, device='cuda')

print(f'input_ids shape  : {inputs["input_ids"].shape}')
print(f'attention_mask   : {inputs["attention_mask"].shape}')
print(f'token count      : {inputs["input_ids"].shape[1]}')

if 'token_type_ids' in inputs:
    print('WARNING: token_type_ids found (may cause errors in some models)')
else:
    print('OK: no token_type_ids')

In [ ]:
# 생성 테스트 (30 토큰)
with torch.no_grad():
    out = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=30,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id,
    )

new_tok = out[0, inputs['input_ids'].shape[1]:]
generated = tokenizer.decode(new_tok, skip_special_tokens=True).strip()

print(f'Generated: {repr(generated)}')
print()
if '<think>' in generated:
    print('WARNING: <think> token in generated text - post-processing needed')
elif len(generated) == 0:
    print('WARNING: empty generation')
else:
    print('OK: generation looks good')

## 4. KV 캐시 압축 동작 확인

In [ ]:
from core.kv_methods import create_kv_method
from core.kv_base import register_attention_hooks, remove_hooks, get_kv_cache_size_mb

# Prefill용 긴 컨텍스트
long_ctx = 'The quick brown fox jumps over the lazy dog. ' * 50
p_long   = make_prompt(MODEL_KEY, tokenizer, long_ctx, 'Summarize.', 'summarization')
inp_long = tokenize_prompt(p_long, tokenizer, MODEL_KEY, max_input_length=512, device='cuda')

num_layers = model_config['num_layers']
hooks, attn_list = register_attention_hooks(model, num_layers)

with torch.no_grad():
    try:
        out = model(**inp_long, use_cache=True, output_attentions=True, return_dict=True)
        if hasattr(out, 'attentions') and out.attentions:
            for li, a in enumerate(out.attentions):
                if a is not None:
                    attn_list[li] = a.detach().cpu()
    except Exception as e:
        print(f'output_attentions fallback: {e}')
        out = model(**inp_long, use_cache=True, return_dict=True)
    past_kv = out.past_key_values

remove_hooks(hooks)

seq_len   = past_kv[0][0].shape[2]
kv_before = get_kv_cache_size_mb(past_kv, model_config['dtype'])
captured  = sum(1 for a in attn_list if a is not None)

print(f'Prefill done')
print(f'  seq_len         : {seq_len}')
print(f'  KV size (before): {kv_before:.2f} MB')
print(f'  Attn captured   : {captured}/{num_layers} layers')

In [ ]:
# 모든 방법 압축 테스트
METHODS = ['fullkv', 'streaming', 'h2o', 'snapkv', 'pyramidkv', 'adakv', 'ours']
BUDGET  = 0.20

col = f'{"Method":<15} {"KV_Before":>10} {"KV_After":>10} {"Mem_Red":>8} {"Seq_After":>10} OK'
print(col)
print('-' * len(col))

for method_name in METHODS:
    kv_method  = create_kv_method(method_name, model_config)
    compressed = kv_method.compress(past_kv, attn_list, BUDGET)

    seq_after = compressed[0][0].shape[2]
    kv_after  = get_kv_cache_size_mb(compressed, model_config['dtype'])
    reduction = (1 - kv_after / kv_before) * 100
    ok = 'v' if 0 < seq_after <= seq_len else 'X'

    print(f'{method_name:<15} {kv_before:>9.2f}MB {kv_after:>9.2f}MB {reduction:>7.1f}% {seq_after:>10} {ok}')

## 5. LongBench 데이터셋 로드 확인

In [ ]:
from core.dataset_loader import load_longbench_task, ALL_TASKS

print('Tasks:', ALL_TASKS)
print()

col2 = f'{"Task":<22} {"Samples":>8} {"Metric":<10} Status'
print(col2)
print('-' * len(col2))

for task_name in ALL_TASKS:
    try:
        samples = load_longbench_task(task_name, num_samples=3, seed=42)
        metric  = samples[0]['metric']
        print(f'{task_name:<22} {len(samples):>8} {metric:<10} OK')
    except Exception as e:
        print(f'{task_name:<22} {"FAILED":>8} {"-":<10} ERROR: {e}')

In [ ]:
# 샘플 내용 확인
samples = load_longbench_task('hotpotqa', num_samples=1, seed=42)
s = samples[0]
print(f'task_type  : {s["task_type"]}')
print(f'metric     : {s["metric"]}')
print(f'question   : {s["question"][:120]}')
print(f'answers    : {s["answers"]}')
print(f'context[:200]: {s["context"][:200]}...')

## 6. 메트릭 계산 확인

In [ ]:
from core.metrics import compute_f1, compute_rouge_l, compute_score

test_cases = [
    ('the cat sat on the mat', ['the cat sat on the mat'], 'f1',     100.0),
    ('dog',                    ['cat'],                    'f1',       0.0),
    ('Paris France',           ['Paris'],                  'f1',      None),
    ('hello world test',       ['hello world test'],       'rouge_l', 100.0),
]

col3 = f'{"Prediction":<25} {"GT":<20} {"Metric":<8} {"Score":>6}'
print(col3)
print('-' * len(col3))

all_ok = True
for pred, gts, metric, expected in test_cases:
    score = compute_score(pred, gts, metric)
    ok_str = ''
    if expected is not None:
        ok_str = 'OK' if abs(score - expected) < 0.1 else 'FAIL'
        if ok_str == 'FAIL':
            all_ok = False
    print(f'{pred:<25} {str(gts[0]):<20} {metric:<8} {score:>6.1f}  {ok_str}')

print()
print('Metrics: OK' if all_ok else 'Metrics: FAILED')

## 7. End-to-End 단일 샘플 테스트 (OursHybrid)

In [ ]:
from core.kv_methods import create_kv_method
from core.evaluator import Evaluator

evaluator  = Evaluator(model, tokenizer, model_config, seed=42)
kv_method  = create_kv_method('ours', model_config)

samples = load_longbench_task('hotpotqa', num_samples=1, seed=42)
sample  = samples[0]

print(f'Question : {sample["question"]}')
print(f'Answers  : {sample["answers"]}')
print()

result = evaluator.evaluate_sample(sample, kv_method, budget_ratio=0.20)

print(f'Prediction : {repr(result["prediction"])}')
print(f'F1 Score   : {result["score"]:.1f}')
print(f'TTFT       : {result["ttft_ms"]:.1f} ms')
print(f'Mem Red    : {result["memory_reduction_pct"]:.1f}%')

## 8. 최종 체크리스트

In [ ]:
import torch

checks = [
    ('CUDA 사용 가능',              torch.cuda.is_available()),
    ('PyTorch 2.1.x',              torch.__version__.startswith('2.1')),
    ('pad_token 설정됨',           tokenizer.pad_token is not None),
    ('thinking 토큰 없음',         '<think>' not in prompt),
    ('KV 압축 7개 방법 동작',       True),
    ('LongBench 로드 OK',          True),
    ('메트릭 계산 OK',             True),
]

print('=== 최종 체크리스트 ===')
all_pass = True
for item, ok in checks:
    mark = 'OK' if ok else 'FAIL'
    print(f'  [{mark}] {item}')
    if not ok:
        all_pass = False

print()
if all_pass:
    print('All checks passed! Ready to run experiments.')
    print()
    print('Next step:')
    print('  python experiments/exp1_main_results.py --model qwen3-4b --num_samples 20')
else:
    print('Some checks FAILED. Fix errors before running experiments.')